# Data Cleaning File for the semester project, ECSE-4840

### GPX reader utility functions

In [ ]:
import argparse
import csv
import math
from datetime import datetime, timezone
from pathlib import Path
import xml.etree.ElementTree as ET

FIELDS = ["time", "elapsed_time_s", "latitude", "longitude", "elevation_m",
          "distance_m", "gradient_percent", "heart_rate_bpm", "cadence_rpm", "power_w"]


def localname(tag):
    return tag.rsplit('}', 1)[-1].lower()


def child_text(element, name):
    for child in element:
        if localname(child.tag) == name:
            return child.text
    return None


def float_or_none(value):
    try:
        return float(value) if value is not None and value.strip() else None
    except (ValueError, AttributeError):
        return None


def parse_time(value):
    if not value:
        return None
    try:
        dt = datetime.fromisoformat(value.strip().replace('Z', '+00:00'))
        return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt
    except ValueError:
        return None


def haversine(lat1, lon1, lat2, lon2):
    r = 6371000.0
    a1, a2 = math.radians(lat1), math.radians(lat2)
    da = a2 - a1
    dl = math.radians(lon2 - lon1)
    a = math.sin(da / 2)**2 + math.cos(a1) * math.cos(a2) * math.sin(dl / 2)**2
    return 2 * r * math.asin(min(1.0, math.sqrt(a)))


def extension_measurements(point):
    values = {"heart_rate_bpm": None, "cadence_rpm": None, "power_w": None}
    aliases = {
        "hr": "heart_rate_bpm", "heartrate": "heart_rate_bpm", "heartratebpm": "heart_rate_bpm",
        "cad": "cadence_rpm", "cadence": "cadence_rpm",
        "power": "power_w", "watts": "power_w", "powerinwatts": "power_w",
    }
    for node in point.iter():
        if node is point:
            continue
        key = aliases.get(localname(node.tag))
        if key and values[key] is None:
            values[key] = float_or_none(node.text)
    return values


def read_gpx(path):
    tree = ET.parse(path)
    root = tree.getroot()
    segments = []
    for node in root.iter():
        if localname(node.tag) == 'trkseg':
            segments.append([p for p in node if localname(p.tag) == 'trkpt'])
    if not segments:
        # Route points are a fallback for GPX route-only files.
        for node in root.iter():
            if localname(node.tag) == 'rte':
                segments.append([p for p in node if localname(p.tag) == 'rtept'])
    if not segments:
        raise ValueError('No GPX track points or route points found.')

    rows = []
    cumulative = 0.0
    start_time = None
    for segment in segments:
        previous = None
        for point in segment:
            lat = float_or_none(point.get('lat'))
            lon = float_or_none(point.get('lon'))
            if lat is None or lon is None:
                continue
            elevation = float_or_none(child_text(point, 'ele'))
            raw_time = child_text(point, 'time')
            timestamp = parse_time(raw_time)
            if start_time is None and timestamp is not None:
                start_time = timestamp
            if previous is not None:
                cumulative += haversine(previous[0], previous[1], lat, lon)
            previous = (lat, lon)
            row = {
                'time': raw_time or '',
                'elapsed_time_s': (timestamp - start_time).total_seconds() if timestamp and start_time else None,
                'latitude': lat, 'longitude': lon, 'elevation_m': elevation,
                'distance_m': cumulative, 'gradient_percent': None,
                **extension_measurements(point),
            }
            rows.append(row)
    return rows



### Read GPX and Organize into Dataframes